In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch

from tqdm import tqdm

In [ ]:
import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

plt.rcParams.update({
    'figure.facecolor': '#282a2c',
    'figure.edgecolor': '#282a2c',
    'axes.facecolor':   '#282a2c',
    'axes.edgecolor':   '#DDE2F4',
    'axes.labelcolor':  '#DDE2F4',
    'xtick.color':      '#DDE2F4',
    'ytick.color':      '#DDE2F4',
    'text.color':       '#DDE2F4',
    'axes.spines.right': False,
    'axes.spines.top':   False,
    'axes.titleweight': 'bold',
    'axes.labelweight': 'bold',
    'savefig.dpi':300,
})

In [ ]:
# GPT2 tokenizer and model
from transformers import GPT2Model
model = GPT2Model.from_pretrained('gpt2')

# the embeddings matrix
embeddings = model.wte.weight.detach() # model.wte is GPT2's token embedding layer. GPT2 has 50257 tokens, each token is represented by 768 numbers.

print(embeddings.shape)

# downsample and reduce the precision before calculations
skip = 5
embeddings = embeddings[::skip,:].to(torch.float16) #keep every fifth token; normally 32-bit. now float16 uses half memory
embeddings.shape

In [ ]:
# cosine similarity between every pai of GPT2 token embeddings
E_norm = torch.nn.functional.normalize(embeddings,p=2,dim=1)

N = embeddings.shape[0]
print(N)

block = 1024 #process 1024 embeddings at a time
csM = torch.empty(N,N,dtype=E_norm.dtype) #allocate similarity matrix

for i in tqdm(range(0,N,block)): #calculate similarites block by block

  end_i = min(i+block,N)

  cs_part = E_norm[i:end_i] @ E_norm.T #compute cosine similarities
  csM[i:end_i] = cs_part #store that block

In [ ]:
# get the non-redundant values of that matrix
row,col = np.triu_indices(csM.shape[0], k=1)
cs_nonredun = csM[row,col]


fig,axs = plt.subplots(1,2,figsize=(12,3.5))

h = axs[0].imshow(csM,vmin=.1,vmax=.4,cmap='plasma')
axs[0].set(title='A) Cosine similarity matrix (GPT-2)',xticks=[],yticks=[],
           xlabel='Tokens',ylabel='Tokens')
plt.colorbar(h,ax=axs[0],pad=.01)

axs[1].hist(cs_nonredun,bins=100,density=True,color=[.9,.7,.9],edgecolor='k')
axs[1].set(xlabel='Cosine similarity',ylabel='Density (log)',
           xlim=[cs_nonredun.min(),cs_nonredun.max()],yscale='log',
           title='B) Distribution of similarities in GPT-2')

plt.tight_layout()
plt.savefig('histogram_similarities_gpt2.png')
plt.show()

In [ ]:
# mean-center and variance-normalize
E_norm = embeddings - embeddings.mean(dim=1,keepdim=True)
E_norm = torch.nn.functional.normalize(E_norm,p=2,dim=1)

R = torch.empty(N,N,dtype=E_norm.dtype)

# loop over blocks
for i in tqdm(range(0,N,block)):

  # find end index
  end_i = min(i+block,N)

  # calculate just this block of correlation and put into matrix
  R_part = E_norm[i:end_i] @ E_norm.T
  R[i:end_i] = R_part


# extract non-redundant matrix elements
R_nonredun = R[row,col]

In [ ]:
# calculate histograms
yAr,xAr = np.histogram(embeddings.mean(dim=1).numpy(),bins=80) # convert to numpy (not actually necessary here)
yL1,xL1 = np.histogram(abs(embeddings).mean(dim=1),bins=80)


fig,axs = plt.subplots(1,2,figsize=(10,4))

# show the scatter plot
skip = 50000
# line of unity
axs[0].axline(np.full(2,R_nonredun[::skip].min().item()),slope=1,
              color='k',linestyle='--',linewidth=.3)
# scatter
axs[0].plot(R_nonredun[::skip],cs_nonredun[::skip],'ko',markerfacecolor=[.7,.7,.9,.3])
axs[0].set(xlabel='Correlation coefficient',ylabel='Cosine similarity',
           title='A) Correlation by similarity')

# and the histograms
axs[1].plot(xAr[:-1],yAr,linewidth=2,label='Arithmetic means')
axs[1].plot(xL1[:-1],yL1,linewidth=2,label='L1 means')
axs[1].legend()
axs[1].set(xlabel='Mean values',ylabel='Count',ylim=[0,None],
           title='B) Distributions of arithmetic vs. L1 means')

plt.tight_layout()
plt.savefig('histogram_corr_distribution.png')
plt.show()